# Lenguajes y Autómatas II (SCD-1016)
## Sesión 05 · Recorridos del AST y Notación Polaca Inversa

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿En qué orden hay que visitar un árbol para poder calcularlo?
>
> **Objetivo de la sesión:** Recorrer un AST en postorden y producir la notación polaca inversa (RPN), el formato que usa una máquina de pila para evaluar.

**Entregable:** tu función postorden en artesano_sesion05_c.c, el harness en verde y la bitácora

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)

> ¿Dudas de C? Consulta la [guía de C](https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2/blob/main/docs/guia-c.md) antes de empezar.


## 1. ¿Qué vas a construir?

Dado un árbol, escribes su recorrido postorden separado por espacios en el buffer que te dan; el harness compara el texto exacto.

Al final de la sesión tendrás: **Una función que convierte cualquier AST en su RPN y la escribe en un buffer, sin imprimir directamente.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 05](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion05.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion05.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion05.h"

void postorden(ASTNode *n, char *buf, int cap);
```

- No imprimas con printf dentro de la función: escribe en el buffer.
- Respeta la capacidad: nunca escribas más de cap-1 caracteres más el terminador.
- Un árbol nulo debe producir cadena vacía.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion05_c.c` junto con `artesano_sesion05_c.c`
> (o `ia_sesion05_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `hoja 42` | `42` |
| 2 | `3+(5*2)` | `3 5 2 * +` |
| 3 | `arbol NULL` | `(cadena vacía)` |
| 4 | `(1+2)*(3+4)` | `1 2 + 3 4 + *` |
| 5 | `8-3` | `8 3 -` |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 05 - Recorrido postorden y RPN ===
Caso                               esperado:              obtenido:              resultado
[ 1] hoja 42                            esperado: 42           obtenido: 42           OK
[ 2] 3+(5*2)                            esperado: 3 5 2 * +    obtenido: 3 5 2 * +    OK
[ 3] arbol NULL                         esperado:              obtenido:              OK
[ 4] (1+2)*(3+4)
     esperado: 1 2 + 3 4 + *
     obtenido: 1 2 + 3 4 + *
     -> OK
[ 5] 8-3                                esperado: 8 3 -        obtenido: 8 3 -        OK

RESULTADO: 5/5 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion05.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] El recorrido respeta el orden izquierdo, derecho, raíz.
- [ ] Los tokens quedan separados por un solo espacio y sin espacio final.
- [ ] Un árbol nulo produce cadena vacía y el buffer siempre termina en '\0'.


## Paso 0 · Entorno


In [9]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


✅ Entorno listo:
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion05_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [10]:
%%writefile api_sesion05.h
/* api_sesion05.h - contrato de la Sesion 05
 * Recorrido postorden y RPN
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION05_H
#define API_SESION05_H

typedef enum { NODO_NUM, NODO_BINOP } TipoNodo;
typedef struct ASTNode {
    TipoNodo tipo;
    int valor;
    char op;
    struct ASTNode *izq;
    struct ASTNode *der;
} ASTNode;

void postorden(ASTNode *n, char *buf, int cap);

#endif /* API_SESION05_H */


Overwriting api_sesion05.h


In [11]:
%%writefile artesano_sesion05_c.c
/* artesano_sesion05_c.c - Fase 1: El Artesano (sin IA)
 * Recorrido postorden y RPN.
 */
#include "api_sesion05.h"
#include <stdio.h>
#include <string.h>

/* agrega un token al final del buffer, con un espacio antes si ya hay algo */
static void pegar(char *buf, int cap, const char *tok) {
    int len = (int)strlen(buf);
    int espacio = (len > 0) ? 1 : 0;
    int nuevo = (int)strlen(tok);

    /* si no cabe (dejando lugar para el '\0'), no lo escribo */
    if (len + espacio + nuevo >= cap) {
        return;
    }
    if (espacio) {
        strcat(buf, " ");
    }
    strcat(buf, tok);
}

static void recorrer(ASTNode *n, char *buf, int cap) {
    if (n == NULL) {
        return;
    }
    recorrer(n->izq, buf, cap);   /* 1. izquierdo */
    recorrer(n->der, buf, cap);   /* 2. derecho */

    /* 3. raiz */
    char tok[16];
    if (n->tipo == NODO_NUM) {
        snprintf(tok, sizeof(tok), "%d", n->valor);
    } else {
        tok[0] = n->op;
        tok[1] = '\0';
    }
    pegar(buf, cap, tok);
}

void postorden(ASTNode *n, char *buf, int cap) {
    if (buf == NULL || cap <= 0) {
        return;
    }
    buf[0] = '\0';   /* arbol nulo = cadena vacia */
    recorrer(n, buf, cap);
}

Overwriting artesano_sesion05_c.c


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion05_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


CONTEXTO: Estoy en la materia Lenguajes y Autómatas II, programando en C (estándar C11).
Tengo un árbol sintáctico abstracto (AST) definido en api_sesion05.h con un enum TipoNodo
(NODO_NUM, NODO_BINOP) y los campos tipo, valor, op, izq y der. Quiero convertirlo a
notación polaca inversa (RPN). Se compila con gcc -std=c11 -Wall -Wextra -Werror, o sea
que cualquier advertencia es error.

ROL: Actúa como ingeniero de software de sistemas y compiladores, con énfasis en
recorridos de árboles y manejo seguro de buffers.

ESPECIFICACIÓN: Implementa exactamente esta firma, sin cambiarla:
  void postorden(ASTNode *n, char *buf, int cap);
Reglas: recorre en postorden (izquierdo, derecho, raíz); escribe los tokens separados por
un solo espacio y sin espacio final; NO uses printf, escribe en el buffer; nunca escribas
más de cap-1 caracteres más el '\0'; un árbol NULL produce cadena vacía; el buffer siempre
debe terminar en '\0'. Ejemplo: 3+(5*2) produce "3 5 2 * +".

OUTPUT: Entrégame solo el archivo .c con #include "api_sesion05.h", sin main(), con
comentarios breves y el análisis de complejidad de la función.

In [12]:
%%writefile ia_sesion05_c.c
/* ia_sesion05_c.c - Fase 2: El Cientifico */
#include "api_sesion05.h"
#include <stdio.h>
#include <string.h>

/* pos guarda cuantos caracteres llevo escritos */
static void emitir(ASTNode *n, char *buf, int cap, int *pos) {
    if (n == NULL) {
        return;
    }
    emitir(n->izq, buf, cap, pos);
    emitir(n->der, buf, cap, pos);

    char tok[16];
    if (n->tipo == NODO_NUM) {
        snprintf(tok, sizeof(tok), "%d", n->valor);
    } else {
        tok[0] = n->op;
        tok[1] = '\0';
    }

    int len = (int)strlen(tok);
    int sep = (*pos > 0) ? 1 : 0;
    if (*pos + sep + len >= cap) {
        return;  /* no cabe */
    }
    if (sep) {
        buf[*pos] = ' ';
        (*pos)++;
    }
    strcpy(buf + *pos, tok);
    *pos += len;
}

void postorden(ASTNode *n, char *buf, int cap) {
    if (buf == NULL || cap <= 0) {
        return;
    }
    buf[0] = '\0';
    int pos = 0;
    emitir(n, buf, cap, &pos);
}

Overwriting ia_sesion05_c.c


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [13]:
%%writefile harness_sesion05_c.c
/* harness_sesion05_c.c - auditoria automatica de la Sesion 05
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion05_c.c artesano_sesion05_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion05.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

static void verificar_str_largo(const char *caso, const char *esperado,
                                const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %s\n", probadas, caso);
    printf("     esperado: %s\n", esperado);
    printf("     obtenido: %s\n", obtenido ? obtenido : "(NULL)");
    printf("     -> %s\n", ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 05 - Recorrido postorden y RPN ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { ASTNode a = {NODO_NUM, 42, 0, NULL, NULL}; char b[64]; postorden(&a, b, 64); verificar_str("hoja 42", "42", b); }
    { ASTNode a = {NODO_NUM, 3, 0, NULL, NULL}; ASTNode b = {NODO_NUM, 5, 0, NULL, NULL}; ASTNode c = {NODO_NUM, 2, 0, NULL, NULL}; ASTNode m = {NODO_BINOP, 0, '*', &b, &c}; ASTNode s = {NODO_BINOP, 0, '+', &a, &m}; char buf[64]; postorden(&s, buf, 64); verificar_str("3+(5*2)", "3 5 2 * +", buf); }
    { char b[16]; postorden(NULL, b, 16); verificar_str("arbol NULL", "", b); }
    { ASTNode a = {NODO_NUM, 1, 0, NULL, NULL}; ASTNode b = {NODO_NUM, 2, 0, NULL, NULL}; ASTNode c = {NODO_NUM, 3, 0, NULL, NULL}; ASTNode d = {NODO_NUM, 4, 0, NULL, NULL}; ASTNode s1 = {NODO_BINOP, 0, '+', &a, &b}; ASTNode s2 = {NODO_BINOP, 0, '+', &c, &d}; ASTNode m = {NODO_BINOP, 0, '*', &s1, &s2}; char buf[64]; postorden(&m, buf, 64); verificar_str_largo("(1+2)*(3+4)", "1 2 + 3 4 + *", buf); }
    { ASTNode a = {NODO_NUM, 8, 0, NULL, NULL}; ASTNode b = {NODO_NUM, 3, 0, NULL, NULL}; ASTNode r = {NODO_BINOP, 0, '-', &a, &b}; char buf[32]; postorden(&r, buf, 32); verificar_str("8-3", "8 3 -", buf); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion05.md" "\n");
    return 0;
}


Writing harness_sesion05_c.c


In [14]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion05_c.c artesano_sesion05_c.c -o test_artesano && ./test_artesano

=== AUDITORIA SESION 05 - Recorrido postorden y RPN ===
Caso                               esperado:              obtenido:              resultado
[ 1] hoja 42                            esperado: 42           obtenido: 42           OK
[ 2] 3+(5*2)                            esperado: 3 5 2 * +    obtenido: 3 5 2 * +    OK
[ 3] arbol NULL                         esperado:              obtenido:              OK
[ 4] (1+2)*(3+4)
     esperado: 1 2 + 3 4 + *
     obtenido: 1 2 + 3 4 + *
     -> OK
[ 5] 8-3                                esperado: 8 3 -        obtenido: 8 3 -        OK

RESULTADO: 5/5 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion05.md


In [15]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion05_c.c ia_sesion05_c.c -o test_ia && ./test_ia

=== AUDITORIA SESION 05 - Recorrido postorden y RPN ===
Caso                               esperado:              obtenido:              resultado
[ 1] hoja 42                            esperado: 42           obtenido: 42           OK
[ 2] 3+(5*2)                            esperado: 3 5 2 * +    obtenido: 3 5 2 * +    OK
[ 3] arbol NULL                         esperado:              obtenido:              OK
[ 4] (1+2)*(3+4)
     esperado: 1 2 + 3 4 + *
     obtenido: 1 2 + 3 4 + *
     -> OK
[ 5] 8-3                                esperado: 8 3 -        obtenido: 8 3 -        OK

RESULTADO: 5/5 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion05.md


## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 5/5 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion05.md` a `auditorias/auditoria_sesion05.md` en tu fork.
2. Adjunta la salida del harness (las 5 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
